# NN Diversity Seat — GCI Home Credit (Phase 2)
## Goal

Every GBM member is 0.97–0.998 rank-correlated with the others, so the ensemble has hit its
ceiling (CV 0.76224). This notebook tests the only remaining model family — **neural networks
with categorical embeddings** — under the exact same protocol as the GBM members:

- Same 98-column engineered features + in-fold target encoding (identical FE code as the
  final pipeline — feature parity guaranteed).
- Same 5-fold StratifiedKFold (**seed 42**), same seeds (42/1337/2026 logic).
- **Seat gate (hard rule):** a candidate earns an ensemble seat only if
  **solo OOF AUC ≥ 0.755** AND **Spearman rank-corr ≤ 0.90 vs XGBoost**.
  Anything below the bar is reported and **rejected — nothing unvalidated gets blended**.

Candidates: **MLP + embeddings** (PyTorch) and **TabNet** (`pytorch-tabnet`), both trained
**from scratch** on competition data only (no pretrained weights — complies with the
no-external-data rule).

Runtime: ~45–70 min on a Colab T4. Artifacts saved to
`/content/drive/MyDrive/GCI Competition/nn_diversity/` and the summary is printed —
send the `.npy` files + `nn_summary.json` back for the verified v4 blend assembly.

**References:** GCI tutorial + README (advice: "explore different models"); pytorch /
pytorch-tabnet docs; standard tabular-MLP practice (embeddings for categorical features).

In [ ]:
# 1. Setup
!pip install -q pytorch-tabnet 2>/dev/null || echo "tabnet install skipped"

import numpy as np, pandas as pd, time, json, os, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import torch, torch.nn as nn

SEED = 42
N_SPLITS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)

In [ ]:
# 2. Data
from google.colab import drive
drive.mount('/content/drive')

COMPETITION_DIR = '/content/drive/MyDrive/competition'   # adjust if needed
MODELS_DIR = '/content/drive/MyDrive/GCI Competition/models'
OUT_DIR = '/content/drive/MyDrive/GCI Competition/nn_diversity'
os.makedirs(OUT_DIR, exist_ok=True)
INPUT = f'{COMPETITION_DIR}/input'

train = pd.read_csv(f'{INPUT}/train.csv')
test = pd.read_csv(f'{INPUT}/test.csv')
sample_sub = pd.read_csv(f'{INPUT}/sample_submission.csv')
print(train.shape, test.shape)

In [ ]:
# 3. Feature engineering — IDENTICAL to the final pipeline (parity with GBM members)
def handle_anomalies(df):
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    if 'OWN_CAR_AGE' in df.columns:
        df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].clip(lower=0)
    return df

def add_missing_indicators(df, cols):
    for c in cols:
        if c in df.columns:
            df[f'{c}_WAS_MISSING'] = df[c].isnull().astype(np.int8)
    return df

def add_domain_ratios(df):
    eps = 1e-5
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['DAYS_EMPLOYED_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + eps)
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + eps)
    df['CREDIT_TO_GOODS_RATIO'] = df['AMT_CREDIT'] / (df['AMT_GOODS_PRICE'] + eps)
    df['ANNUITY_TO_CREDIT_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['INCOME_TO_CREDIT_RATIO'] = df['AMT_INCOME_TOTAL'] / (df['AMT_CREDIT'] + eps)
    df['PHONE_CHANGE_TO_AGE'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_BIRTH'] + eps)
    df['EMPLOYED_TO_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['REGISTRATION_TO_BIRTH'] = df['DAYS_REGISTRATION'] / (df['DAYS_BIRTH'] + eps)
    df['ID_PUBLISH_TO_BIRTH'] = df['DAYS_ID_PUBLISH'] / (df['DAYS_BIRTH'] + eps)
    df['CAR_AGE_TO_EMPLOYED'] = df['OWN_CAR_AGE'] / (df['DAYS_EMPLOYED'].abs() + eps)
    df['CAR_AGE_TO_BIRTH'] = df['OWN_CAR_AGE'] / (df['DAYS_BIRTH'].abs() + eps)
    return df

def add_ext_source_interactions(df):
    ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCES_MEAN'] = df[ext].mean(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_SOURCES_STD'] = df[ext].std(axis=1)
    df['EXT_SOURCES_MAX'] = df[ext].max(axis=1)
    df['EXT_SOURCES_MIN'] = df[ext].min(axis=1)
    df['EXT_1_2_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0)
    df['EXT_1_3_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_2_3_PROD'] = df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_1_2_DIFF'] = (df['EXT_SOURCE_1'] - df['EXT_SOURCE_2']).abs()
    df['EXT_1_3_DIFF'] = (df['EXT_SOURCE_1'] - df['EXT_SOURCE_3']).abs()
    df['EXT_2_3_DIFF'] = (df['EXT_SOURCE_2'] - df['EXT_SOURCE_3']).abs()
    df['EXT_1_2_RATIO'] = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_2'] + 1e-5)
    df['EXT_1_3_RATIO'] = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_3'] + 1e-5)
    df['EXT_2_3_RATIO'] = df['EXT_SOURCE_2'] / (df['EXT_SOURCE_3'] + 1e-5)
    df['EXT_SOURCES_COUNT'] = df[ext].notnull().sum(axis=1)
    df['EXT_SOURCES_WEIGHTED_MEAN'] = (df['EXT_SOURCE_1'].fillna(0) + df['EXT_SOURCE_2'].fillna(0) + df['EXT_SOURCE_3'].fillna(0)) / (df['EXT_SOURCES_COUNT'] + 1e-5)
    return df

def add_bureau_aggregations(df):
    b = ['AMT_REQ_CREDIT_BUREAU_HOUR','AMT_REQ_CREDIT_BUREAU_MON',
         'AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR']
    df['BUREAU_QUERY_SUM'] = df[b].fillna(0).sum(axis=1)
    df['BUREAU_QUERY_MEAN'] = df[b].fillna(0).mean(axis=1)
    df['BUREAU_QUERY_MAX'] = df[b].fillna(0).max(axis=1)
    df['BUREAU_QUERY_COUNT'] = df[b].notnull().sum(axis=1)
    df['BUREAU_RECENT_TO_HIST_RATIO'] = (df['AMT_REQ_CREDIT_BUREAU_HOUR'].fillna(0) + df['AMT_REQ_CREDIT_BUREAU_MON'].fillna(0)) / (df['AMT_REQ_CREDIT_BUREAU_QRT'].fillna(0) + df['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(0) + 1e-5)
    return df

def add_binned_features(df, ref):
    binc = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY',
            'DAYS_BIRTH','DAYS_EMPLOYED','DAYS_REGISTRATION','DAYS_ID_PUBLISH','DAYS_LAST_PHONE_CHANGE',
            'CNT_FAM_MEMBERS','CNT_CHILDREN','REGION_POPULATION_RELATIVE','REGION_RATING_CLIENT']
    for col in binc:
        if col in df.columns:
            try:
                q = ref[col].quantile([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]).values
                df[f'{col}_QBIN'] = pd.cut(df[col], bins=[-np.inf]+list(q)+[np.inf], labels=False, duplicates='drop')
            except Exception:
                df[f'{col}_QBIN'] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')
            df[f'{col}_QBIN'] = df[f'{col}_QBIN'].astype(np.int8)
    return df

HIGH_MISSING = ['EXT_SOURCE_1','OWN_CAR_AGE','EXT_SOURCE_3','AMT_REQ_CREDIT_BUREAU_HOUR',
                'AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR',
                'EXT_SOURCE_2','CNT_FAM_MEMBERS']

y = train['TARGET'].values
train_ids = train['SK_ID_CURR'].values
test_ids = test['SK_ID_CURR'].values

tr_fe = train.drop(columns=['TARGET','SK_ID_CURR']).copy()
te_fe = test.drop(columns=['SK_ID_CURR']).copy()

tr_fe = handle_anomalies(tr_fe); te_fe = handle_anomalies(te_fe)
tr_fe = add_missing_indicators(tr_fe, HIGH_MISSING); te_fe = add_missing_indicators(te_fe, HIGH_MISSING)
tr_fe = add_domain_ratios(tr_fe); te_fe = add_domain_ratios(te_fe)
tr_fe = add_ext_source_interactions(tr_fe); te_fe = add_ext_source_interactions(te_fe)
tr_fe = add_bureau_aggregations(tr_fe); te_fe = add_bureau_aggregations(te_fe)
tr_fe = add_binned_features(tr_fe, tr_fe); te_fe = add_binned_features(te_fe, tr_fe)

cat_cols = tr_fe.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    combined = pd.concat([tr_fe[col], te_fe[col]], axis=0)
    le = LabelEncoder(); le.fit(combined.astype(str))
    tr_fe[col] = le.transform(tr_fe[col].astype(str))
    te_fe[col] = le.transform(te_fe[col].astype(str))
    freq = combined.value_counts(normalize=True).to_dict()
    tr_fe[f'{col}_FREQ'] = tr_fe[col].map(freq).astype(np.float32)
    te_fe[f'{col}_FREQ'] = te_fe[col].map(freq).astype(np.float32)

feature_cols = list(tr_fe.columns)
tr_fe[feature_cols] = tr_fe[feature_cols].fillna(-999)
te_fe[feature_cols] = te_fe[feature_cols].fillna(-999)
X0 = tr_fe.astype(np.float32)
X_test0 = te_fe.astype(np.float32)
print(f"engineered: {X0.shape}, {X_test0.shape}")

In [ ]:
# 4. Enrichment + dead-feature removal (same as final pipeline)
DEAD = ['FLAG_MOBIL','ORGANIZATION_TYPE','EXT_SOURCE_2_WAS_MISSING','CNT_FAM_MEMBERS_WAS_MISSING',
        'AMT_REQ_CREDIT_BUREAU_MON_WAS_MISSING','AMT_REQ_CREDIT_BUREAU_YEAR_WAS_MISSING',
        'EXT_SOURCE_1_WAS_MISSING','EXT_SOURCE_3_WAS_MISSING','FLAG_EMP_PHONE',
        'EXT_SOURCE_3_QBIN','OCCUPATION_TYPE_FREQ','NAME_HOUSING_TYPE_FREQ','NAME_FAMILY_STATUS_FREQ',
        'NAME_EDUCATION_TYPE_FREQ','NAME_INCOME_TYPE_FREQ','CODE_GENDER_FREQ','NAME_CONTRACT_TYPE_FREQ',
        'ORGANIZATION_TYPE_FREQ']

def enrich(df, base_cols):
    X = df.copy(); eps = 1e-5
    ext = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
    filled = X[ext].replace(-999, np.nan)
    f = filled.fillna(filled.mean())
    age_years = (X['DAYS_BIRTH'].replace(-999, np.nan).abs() / 365.0).fillna(
        X['DAYS_BIRTH'].replace(-999, np.nan).abs().mean() / 365.0)
    for c in ext:
        X[f'{c}X_AGE'] = f[c] * age_years
    X['EXT_MEAN_X_AGE'] = f.mean(axis=1) * age_years
    X['EXT_2x3'] = f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_1x2x3_NANSAFE'] = f['EXT_SOURCE_1'] * f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_2_SQ'] = f['EXT_SOURCE_2'] ** 2
    X['EXT_3_SQ'] = f['EXT_SOURCE_3'] ** 2
    X['EXT_SUM'] = f.sum(axis=1)
    emp_years = X['DAYS_EMPLOYED'].replace(-999, np.nan).abs() / 365.0
    X['CREDIT_PER_INCOME_YEAR'] = X['AMT_CREDIT'] / ((X['AMT_INCOME_TOTAL'] + eps) * emp_years.fillna(emp_years.mean()) + eps)
    X['ANNUITY_PER_FAM'] = X['AMT_ANNUITY'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['CREDIT_PER_FAM'] = X['AMT_CREDIT'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['INCOME_PER_CHILD'] = X['AMT_INCOME_TOTAL'] / (X['CNT_CHILDREN'].clip(lower=0) + 1)
    X['NAN_COUNT'] = (df[base_cols] == -999).sum(axis=1).astype(np.int16)
    X['REGION_RATING_X_EXT_MEAN'] = X['REGION_RATING_CLIENT'] * f.mean(axis=1)
    return X

X = enrich(X0, feature_cols).drop(columns=[c for c in DEAD if c in X0.columns])
X_test = enrich(X_test0, feature_cols).drop(columns=[c for c in DEAD if c in X_test0.columns])
assert list(X.columns) == list(X_test.columns)
assert X.shape == (171202, 98)
print(f"final: {X.shape[1]} features — parity with GBM pipeline confirmed")

In [ ]:
# 5. Folds + in-fold target encoding (same as GBM members)
HIGH_CARD = ['ORGANIZATION_TYPE','OCCUPATION_TYPE','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
             'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','NAME_CONTRACT_TYPE']
TE_COLS = [c for c in HIGH_CARD if c in X.columns]
CATS = ['NAME_CONTRACT_TYPE','CODE_GENDER','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','OCCUPATION_TYPE','FLAG_WORK_PHONE']

def add_te(X_tr, X_va, X_te, y_tr, smoothing=10):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    gs = pd.Series(y_tr, index=X_tr.index); gm = gs.mean()
    for col in TE_COLS:
        means = gs.groupby(X_tr[col]).mean()
        counts = X_tr[col].value_counts()
        sm = (means * counts + gm * smoothing) / (counts + smoothing)
        X_tr[f'{col}_TE'] = X_tr[col].map(sm).fillna(gm)
        X_va[f'{col}_TE'] = X_va[col].map(sm).fillna(gm)
        X_te[f'{col}_TE'] = X_te[col].map(sm).fillna(gm)
    return X_tr, X_va, X_te

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))
fold_data = []
for tri, vai in FOLDS:
    X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
    fold_data.append((tri, vai, X_tr, X_va, X_te))
print('folds ready | TE cols:', TE_COLS)

In [ ]:
# 6. NN preprocessing: embeddings for categoricals, median-impute + standardize for continuous
CONT_COLS = [c for c in fold_data[0][2].columns if c not in CATS]
# embedding cardinality must cover train AND test codes
CARDS = {c: int(max(X[c].max(), X_test[c].max())) + 1 for c in CATS}
print('cards:', CARDS)
print('n_cont:', len(CONT_COLS))

def nn_prep(X_tr, X_va, X_te):
    cont_tr = X_tr[CONT_COLS].replace(-999, np.nan).astype(np.float64)
    med = cont_tr.median()
    filled = cont_tr.fillna(med)
    mu, sd = filled.mean(), filled.std().replace(0, 1.0)
    def pp(df):
        a = df[CONT_COLS].replace(-999, np.nan).astype(np.float64)
        a = a.fillna(med).fillna(0.0)
        a = ((a - mu) / sd).replace([np.inf, -np.inf], 0.0).fillna(0.0)
        return a.astype(np.float32).values
    cont = {'tr': pp(X_tr), 'va': pp(X_va), 'te': pp(X_te)}
    cat = {k: d[CATS].values.astype(np.int64) for k, d in [('tr', X_tr), ('va', X_va), ('te', X_te)]}
    return cat, cont
print('prep ready')

In [ ]:
# 7. MLP with categorical embeddings
class TabularMLP(nn.Module):
    def __init__(self, cards, n_cont, hidden=(512, 256, 128), emb_max=10, p=0.3):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(c, min(emb_max, (c + 1) // 2 + 1)) for c in cards.values()])
        emb_dim = sum(e.embedding_dim for e in self.embs)
        dims = [emb_dim + n_cont] + list(hidden)
        layers = []
        for i in range(len(dims) - 1):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.BatchNorm1d(dims[i+1]), nn.ReLU(), nn.Dropout(p)]
        layers += [nn.Linear(dims[-1], 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, xc, xn):
        e = torch.cat([emb(xc[:, i]) for i, emb in enumerate(self.embs)], dim=1)
        return self.net(torch.cat([e, xn], dim=1)).squeeze(1)

def train_mlp_fold(tri, vai, X_tr, X_va, X_te, seed, max_epochs=50, patience=6, lr=1e-3, batch=1024):
    torch.manual_seed(seed); np.random.seed(seed)
    cat, cont = nn_prep(X_tr, X_va, X_te)
    xc = {k: torch.tensor(cat[k], dtype=torch.long).to(DEVICE) for k in cat}
    xn = {k: torch.tensor(cont[k], dtype=torch.float32).to(DEVICE) for k in cont}
    yt = torch.tensor(y[tri], dtype=torch.float32).to(DEVICE)
    model = TabularMLP(CARDS, len(CONT_COLS)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    lossf = nn.BCEWithLogitsLoss()
    n = len(yt); best_auc, bad, best = 0.0, 0, None
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            opt.zero_grad()
            loss = lossf(model(xc['tr'][idx], xn['tr'][idx]), yt[idx])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vlog = model(xc['va'], xn['va']).cpu().numpy()
        auc = roc_auc_score(y[vai], vlog)
        sched.step(auc)
        if auc > best_auc:
            best_auc, bad = auc, 0
            with torch.no_grad():
                best = (1/(1+np.exp(-vlog)),
                        torch.sigmoid(model(xc['te'], xn['te'])).cpu().numpy())
        else:
            bad += 1
            if bad >= patience: break
    return best_auc, best[0], best[1]

t0 = time.time()
oof_mlp = np.zeros(len(X)); test_mlp = np.zeros(len(X_test))
for fold, (tri, vai, X_tr, X_va, X_te) in enumerate(fold_data):
    a, vo, tp = train_mlp_fold(tri, vai, X_tr, X_va, X_te, seed=SEED + fold)
    oof_mlp[vai] = vo; test_mlp += tp / N_SPLITS
    print(f'MLP fold {fold+1}: val AUC {a:.5f} ({time.time()-t0:.0f}s)', flush=True)
auc_mlp = roc_auc_score(y, oof_mlp)
print(f'MLP OOF AUC: {auc_mlp:.5f}  total {time.time()-t0:.0f}s')

In [ ]:
# 8. TabNet (guarded — MLP is the primary candidate)
RUN_TABNET = True
oof_tab = None
if RUN_TABNET:
    try:
        from pytorch_tabnet.tab_model import TabNetClassifier
        cat_idxs = [list(fold_data[0][2].columns).index(c) for c in CATS]
        feats = fold_data[0][2].columns
        cat_dims = [CARDS[c] for c in CATS]
        cat_emb = [min(10, (d + 1) // 2 + 1) for d in cat_dims]
        oof_tab = np.zeros(len(X)); test_tab = np.zeros(len(X_test))
        t0 = time.time()
        for fold, (tri, vai, X_tr, X_va, X_te) in enumerate(fold_data):
            cat, cont = nn_prep(X_tr, X_va, X_te)
            mtr = np.hstack([cat['tr'].astype(np.float32), cont['tr']])
            mva = np.hstack([cat['va'].astype(np.float32), cont['va']])
            mte = np.hstack([cat['te'].astype(np.float32), cont['te']])
            clf = TabNetClassifier(n_d=32, n_a=32, n_steps=5, gamma=1.3,
                                   cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=cat_emb,
                                   lambda_sparse=1e-3, seed=SEED + fold,
                                   verbose=0, device_name=DEVICE)
            clf.fit(mtr, y[tri], eval_set=[(mva, y[vai])], eval_metric=['auc'],
                    max_epochs=60, patience=8, batch_size=4096, virtual_batch_size=512)
            oof_tab[vai] = clf.predict_proba(mva)[:, 1]
            test_tab += clf.predict_proba(mte)[:, 1] / N_SPLITS
            print(f'TabNet fold {fold+1}: {roc_auc_score(y[vai], oof_tab[vai]):.5f} ({time.time()-t0:.0f}s)', flush=True)
        auc_tab = roc_auc_score(y, oof_tab)
        print(f'TabNet OOF AUC: {auc_tab:.5f}')
    except Exception as ex:
        print('TabNet skipped:', ex)

In [ ]:
# 9. SEAT GATE: solo AUC >= 0.755 AND rank-corr <= 0.90 vs XGBoost
cands = {'MLP': (oof_mlp, test_mlp, auc_mlp)}
if oof_tab is not None:
    cands['TabNet'] = (oof_tab, test_tab, auc_tab)

# try to load the XGB OOF from the earlier Colab ensemble run (for rank-corr)
xgb_ref = None
try:
    xgb_ref = np.load(f'{MODELS_DIR}/v2_oof_xgb.npy')
    print('rank-corr reference: Colab v2 XGB OOF (approx. XGBnew)')
except Exception:
    print('no XGB OOF on Drive — corr check will be done at v4 assembly on verified OOFs')

summary = {'date': time.strftime('%Y-%m-%d %H:%M'), 'device': DEVICE,
           'torch': torch.__version__, 'gate': 'solo>=0.755 AND rankcorr<=0.90', 'candidates': {}}
for name, (o, t, a) in cands.items():
    row = {'oof_auc': round(float(a), 5), 'solo_pass': bool(a >= 0.755)}
    if xgb_ref is not None:
        rc = float(pd.Series(o).corr(pd.Series(xgb_ref), method='spearman'))
        row['rankcorr_vs_xgb'] = round(rc, 4)
        row['corr_pass'] = bool(rc <= 0.90)
    row['SEAT'] = row['solo_pass'] and row.get('corr_pass', 'deferred')
    summary['candidates'][name] = row
    print(f"{name}: OOF {a:.5f} | solo_pass {row['solo_pass']} | corr {row.get('rankcorr_vs_xgb','deferred')} | SEAT: {row['SEAT']}")

In [ ]:
# 10. Save artifacts + (optional) blend preview with the Colab v2 OOFs
np.save(f'{OUT_DIR}/mlp_oof.npy', oof_mlp); np.save(f'{OUT_DIR}/mlp_test.npy', test_mlp)
if oof_tab is not None:
    np.save(f'{OUT_DIR}/tabnet_oof.npy', oof_tab); np.save(f'{OUT_DIR}/tabnet_test.npy', test_tab)
with open(f'{OUT_DIR}/nn_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('saved to', OUT_DIR)

# OPTIONAL preview only: combines with the earlier Colab v2 OOFs (note: that run's CAT was
# GPU-weakened). FINAL v4 blending happens on the locally verified OOFs.
try:
    lgb_o = np.load(f'{MODELS_DIR}/v2_oof_lgb.npy')
    xgb_o = np.load(f'{MODELS_DIR}/v2_oof_xgb.npy')
    cat_o = np.load(f'{MODELS_DIR}/v2_oof_cat.npy')
    base_best = (0.0, None)
    mem = {'LGB': lgb_o, 'XGB': xgb_o, 'CAT': cat_o, 'MLP': oof_mlp}
    if oof_tab is not None: mem['Tab'] = oof_tab
    keys = list(mem.keys())
    for w1 in np.arange(0, 1.0001, 0.05):
        for w2 in np.arange(0, 1.0001 - w1, 0.05):
            for w3 in np.arange(0, 1.0001 - w1 - w2 + 1e-9, 0.05):
                w4 = 1 - w1 - w2 - w3
                if w4 < -1e-9: continue
                pred = w1*mem[keys[0]] + w2*mem[keys[1]] + w3*mem[keys[2]] + w4*mem[keys[3]]
                a = roc_auc_score(y, pred)
                if a > base_best[0]: base_best = (a, (round(w1,2), round(w2,2), round(w3,2), round(w4,2)))
    print(f'PREVIEW blend best (Colab OOFs, incl. NN): {dict(zip(keys, base_best[1]))} -> {base_best[0]:.5f}')
    print('caveat: preview OOFs differ slightly from verified local ones; final call at assembly')
except Exception as ex:
    print('preview skipped:', ex)